# TP3 : Réponses sourcées et refus

**Contexte :** assistant du service client de Lagune Banque.

Retrouver les bons chunks ne suffit pas : l'assistant doit répondre à partir d'eux, **prouver** ce qu'il avance et **refuser** quand la réponse n'est pas dans les documents. Vous allez ajouter deux garde-fous à la génération : un **seuil de refus**, calibré sur des questions étiquetées, et la **vérification des citations**, qui contrôle que chaque extrait figure dans le chunk source.

Complétez les `# TODO`.

## <a id="syntaxe"></a>Syntaxe

Un schéma Pydantic décrit la forme attendue d'une réponse :

```python
class Weather(BaseModel):
    city: str
    days: list[int]
```

Pour une sortie structurée, on passe le schéma à `response_format` : le modèle répond en JSON conforme et Pydantic le valide.

```python
r = litellm.completion(model=..., messages=[{"role": "user", "content": "..."}], response_format=Weather)
weather = Weather.model_validate_json(r.choices[0].message.content)   # ValueError si le JSON ne respecte pas le schéma
```

## <a id="configuration"></a>Configuration

In [8]:
import os
import sys
from pathlib import Path

import litellm
from pydantic import BaseModel
from typing import List

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage")] if (p / "rag.py").exists()).resolve()))
import rag
import rag_advanced as ra
from config import prepare
from weaviate_store import connect

In [2]:
prepare(embedding=True, rerank=True)

Connexion OK : gemini/gemini-3.5-flash-lite gemini/gemini-embedding-2 azure_ai/Cohere-rerank-v4.0-fast


## <a id="recherche"></a>Recherche

L'index est celui du TP1 (récursif, 500 caractères, overlap 75). La recherche est celle du TP2 : 20 candidats vectoriels, reclassés par le reranker, dont on garde les 3 meilleurs. Chaque chunk porte son `score` de reranker, entre 0 et 1.

In [3]:
documents = rag.load_corpus()
for d in documents:
    if d["fichier"].endswith(".pdf") and not d["avertissement"]:
        d["pages"] = rag.strip_headers(d["pages"])
index = rag.index_chunks(connect(), "index_sources", rag.split_documents(documents, 500, 75, "sources"))


def retrieve(question):
    return rag.rerank(question, rag.search(index, question, 20, rag.AUJOURDHUI), 3)


for p in retrieve("Quels sont les frais de retard du Prêt Express ?"):
    print(f"{p['score']:.2f} | {p['meta']['source']} | {p['texte'][:90]!r}")

0.93 | DOC-01 | 'transaction Lagune Money : statut, montant, date et heure, et rattachement\néventuel à une '
0.91 | DOC-08 | 'Délai de décision du conseiller | 24 heures\nFrais de retard (Prêt Express) | 2 000 FCFA\nFr'
0.91 | DOC-05 | "4. Retard et impayés\n\nQue se passe-t-il en cas d'échéance impayée ?\n\nDes frais de retard f"


## <a id="seuil-refus"></a>Seuil de refus

Si même le meilleur chunk est peu pertinent, la question ne correspond à rien dans le corpus : mieux vaut refuser que laisser le modèle improviser. Deux signaux sont disponibles pour décider : la **distance** du meilleur chunk dans l'index vectoriel, et son **score** de reranker. Comparez-les sur des questions étiquetées (dans le corpus ou non).

In [4]:
CALIBRATION = [
    ("Quel est le montant minimum d'un Prêt Express ?", True), ("Quels sont les frais de retard du Prêt Express ?", True),
    ("Quand les intérêts de l'Épargne Plus sont-ils versés ?", True), ("Que faire si mon KYC est incomplet ?", True),
    ("Le service client est-il joignable le dimanche ?", True), ("Quelle est la procédure en cas de suspicion de fraude ?", True),
    ("Quelle est la capitale de la France ?", False), ("Quel est le cours de l'euro aujourd'hui ?", False),
    ("Qui a gagné la dernière Coupe d'Afrique des nations ?", False), ("Comment cuisiner un attiéké ?", False),
    ("Quel est le taux du livret A réglementé ?", False),
]

mesures = []
for question, dans_le_corpus in CALIBRATION:
    candidats = rag.search(index, question, 20, rag.AUJOURDHUI)
    mesures.append((dans_le_corpus, candidats[0]["distance"], rag.rerank(question, candidats, 1)[0]["score"]))

for nom, position in (("distance", 1), ("score du reranker", 2)):
    dans = [m[position] for m in mesures if m[0]]
    hors = [m[position] for m in mesures if not m[0]]
    print(f"{nom:18s} dans le corpus : {min(dans):.2f} à {max(dans):.2f} | hors corpus : {min(hors):.2f} à {max(hors):.2f}")

distance           dans le corpus : 0.46 à 0.65 | hors corpus : 0.66 à 0.97
score du reranker  dans le corpus : 0.88 à 0.93 | hors corpus : 0.10 à 0.39


Les plages de valeurs des deux groupes de questions se chevauchent pour la distance, et sont séparées pour le score du reranker : le reranker lit la question et le chunk ensemble, alors que la distance compare deux vecteurs calculés séparément.

`rag.best_threshold` cherche le seuil qui classe le mieux un jeu étiqueté. Il attend des distances (petit = proche) : pour un score (grand = pertinent), on lui passe `1 - score`.

**TODO 1** : calculez le seuil de score. Un chunk dont le score est inférieur à `SEUIL_SCORE` est refusé.

In [6]:
dans = [1 - m[2] for m in mesures if m[0]]
hors = [1 - m[2] for m in mesures if not m[0]]
# TODO 1 : seuil, exactitude = rag.best_threshold(...) puis SEUIL_SCORE = 1 - seuil
SEUIL_SCORE = 0.0
seuil, exactitude = rag.best_threshold(dans, hors)
SEUIL_SCORE = 1 - seuil
print(f"Seuil de score : {SEUIL_SCORE}")

Seuil de score : 0.6373329999999999


## <a id="citations"></a>Vérification des citations

Un `[DOC-06]` placé à la fin d'une phrase ne prouve rien : le modèle peut citer un document qui ne dit pas cela. On demande donc au modèle une sortie structurée (un schéma Pydantic, validé par la bibliothèque) : sa réponse, et pour chaque affirmation la source et l'**extrait exact** qui l'appuie. Le code vérifie ensuite que chaque extrait existe vraiment dans le chunk cité.

**TODO 2** : écrivez les schémas Pydantic `Citation` (`source` et `extrait`, deux `str`) et `SourcedAnswer` (`reponse` : `str`, `citations` : liste de `Citation`).

In [9]:
# TODO 2 : class Citation(BaseModel) et class SourcedAnswer(BaseModel)
class Citation(BaseModel):
    source: str
    extrait: str

class SourcedAnswer(BaseModel):
    reponse: str
    citations: List[Citation]

In [10]:
PROMPT = """Tu es l'assistant du service client de Lagune Banque.
Réponds UNIQUEMENT à partir des chunks fournis, en 2 à 4 phrases. N'invente aucun chiffre, délai ou condition.
Si l'information n'y figure pas, mets "reponse" à "REFUS" et laisse "citations" vide.
Pour chaque citation, "source" est l'identifiant seul (DOC-xx, sans la version) et "extrait" un texte copié mot pour mot depuis le chunk.

# Chunks
{chunks}

# Question
{question}
"""

**TODO 3** : dans `ask_model(question, chunks)`, complétez l'appel à `litellm.completion` : le prompt formaté (`prompt`) en message utilisateur, `response_format=SourcedAnswer`, `model=os.environ["MODELE"]`, `api_key=os.environ["LLM_API_KEY"]`, `temperature=0` et `max_tokens=900`. La validation est fournie : `SourcedAnswer.model_validate_json` lit la réponse, et la fonction renvoie `None` si le modèle ne respecte pas le schéma (`ValueError`).

In [20]:
def ask_model(question, chunks):
    prompt = PROMPT.format(chunks=rag.format_chunks(chunks), question=question)
    # TODO 3 : r = litellm.completion(...) avec prompt en message utilisateur et response_format=SourcedAnswer
    r = litellm.completion(
        messages=[{"role": "user", "content": prompt}],
        response_format=SourcedAnswer,
        model=os.environ["MODELE"],
        api_key=os.environ["LLM_API_KEY"],
        temperature=0.0,
        max_tokens=900
    )
    if r is None:
        return None
    try:
        return SourcedAnswer.model_validate_json(r.choices[0].message.content or "")
    except ValueError:    # le modèle n'a pas respecté le schéma
        return None


question = "Quels sont les frais de retard du Prêt Express ?"
chunks = retrieve(question)
ask_model(question, chunks)

SourcedAnswer(reponse="En cas d'impayé sur un Prêt Express, des frais de retard forfaitaires de 2 000 FCFA s'appliquent. Ceux-ci s'ajoutent au solde restant dû et peuvent être annulés si l'impayé fait l'objet d'une régularisation validée.", citations=[Citation(source='DOC-01', extrait="En cas d'impayé, des frais de retard forfaitaires s'appliquent : 2 000 FCFA pour un Prêt Express")])

In [21]:
question = "Quel est le montant minimum d'un Prêt Express ?"
chunks = retrieve(question)
ask_model(question, chunks)

SourcedAnswer(reponse="Le montant minimum d'un Prêt Express est de 5 000 FCFA. Ce montant est compris entre 5 000 FCFA et 50 000 FCFA.", citations=[Citation(source='DOC-02', extrait='Le montant minimum est de 5 000 FCFA.'), Citation(source='DOC-02', extrait="Le montant d'un Prêt Express est compris entre 5 000 FCFA et 50 000 FCFA.")])

`verify_citations(citations, chunks)` est fournie : elle renvoie les citations **valides**, celles dont l'`extrait` figure dans le texte d'un chunk de la même `source`. La casse, les espaces et les retours à la ligne sont normalisés avant la comparaison, car le modèle les recopie rarement à l'identique. `citations` est une liste de dicts.

In [22]:
def normalize(texte):
    return " ".join(texte.split()).lower()


def verify_citations(citations, chunks):
    return [c for c in citations
            if any(p["meta"]["source"] == c["source"] and normalize(c["extrait"]) in normalize(p["texte"]) for p in chunks)]


sortie = ask_model(question, chunks)
verify_citations([c.model_dump() for c in sortie.citations], chunks) if sortie else []

[{'source': 'DOC-02', 'extrait': 'Le montant minimum est de 5 000 FCFA.'},
 {'source': 'DOC-02',
  'extrait': "Le montant d'un Prêt Express est compris entre 5 000 FCFA et 50 000 FCFA."}]

## <a id="fonction-reponse"></a>Fonction de réponse

`answer` enchaîne les trois étapes : le refus par le score, la génération structurée, la vérification. Une réponse dont **aucune** citation n'est vérifiée est traitée comme un refus : elle n'est pas prouvée.

```mermaid
flowchart TD
    Q["Question"] --> R["Recherche : 3 chunks"]
    R --> S{"Score du meilleur chunk sous le seuil ?"}
    S -->|oui| X["Refus"]
    S -->|non| G["Réponse et citations"]
    G --> V["Vérification des citations"]
    V --> C{"Au moins une citation vérifiée ?"}
    C -->|non| X
    C -->|oui| A["Réponse avec citations"]
```

In [23]:
def answer(question):
    chunks = retrieve(question)
    if not chunks or chunks[0]["score"] < SEUIL_SCORE:
        return {"question": question, "reponse": rag.REFUS, "refus": True, "citations": [], "chunks": chunks, "score": chunks[0]["score"] if chunks else 0}
    sortie = ask_model(question, chunks)
    citations = verify_citations([c.model_dump() for c in sortie.citations], chunks) if sortie else []
    if sortie is None or sortie.reponse.strip() == "REFUS" or not citations:
        return {"question": question, "reponse": rag.REFUS, "refus": True, "citations": [], "chunks": chunks, "score": chunks[0]["score"]}
    return {"question": question, "reponse": sortie.reponse, "refus": False, "citations": citations, "chunks": chunks, "score": chunks[0]["score"]}

## <a id="questions"></a>Jeu de questions

Les réponses aux dix questions figurent dans les documents. Les trois pièges sont plausibles pour une banque mais absents du corpus : le système doit refuser au lieu d'inventer. Le deuxième est le plus difficile : il parle de prêt, donc de sujets que le corpus connaît, mais demande une décision que le corpus ne peut pas donner. Aucun seuil ne le refusera de façon fiable.

In [24]:
QUESTIONS_METIER = [
    "Quel est le montant minimum d'un Prêt Express ?",
    "Quels sont les frais de service d'un Prêt Express+ sur 60 jours ?",
    "Quand les intérêts de l'Épargne Plus sont-ils versés ?",
    "Quel est le taux de rémunération de l'Épargne Plus ?",
    "Quels sont les frais de retard du Prêt Express ?",
    "Combien de temps faut-il attendre pour emprunter à nouveau après un remboursement ?",
    "Que faire si mon KYC est incomplet ?",
    "Que fait l'assistant en cas de suspicion de fraude ?",
    "Le service client est-il joignable le dimanche ?",
    "Mon remboursement a réussi mais il n'est pas rattaché à mon échéance à cause d'un incident : que se passe-t-il ?",
]
QUESTIONS_PIEGES = [
    "Proposez-vous des comptes ou des portefeuilles en cryptomonnaie ?",
    "Pouvez-vous m'accorder un prêt de 200 000 FCFA dès aujourd'hui ?",
    "Quelle est la météo prévue à Abidjan demain ?",
]


def show(r):
    print("QUESTION :", r["question"])
    print("RÉPONSE  :", r["reponse"][:420].replace("\n", " "))
    print("STATUT   :", f"refusé (score {r['score']:.2f})" if r["refus"] else "preuves : " + " | ".join(f"{c['source']} « {c['extrait'][:60]} »" for c in r["citations"]))
    print("-" * 100)


resultats = [answer(q) for q in QUESTIONS_METIER]
pieges = [answer(q) for q in QUESTIONS_PIEGES]
for r in resultats + pieges:
    show(r)
print(f"Questions métier répondues : {sum(not r['refus'] for r in resultats)}/{len(resultats)} | pièges refusés : {sum(r['refus'] for r in pieges)}/{len(pieges)}")

QUESTION : Quel est le montant minimum d'un Prêt Express ?
RÉPONSE  : Le montant minimum d'un Prêt Express est de 5 000 FCFA.
STATUT   : preuves : DOC-02 « Le montant minimum est de 5 000 FCFA. »
----------------------------------------------------------------------------------------------------
QUESTION : Quels sont les frais de service d'un Prêt Express+ sur 60 jours ?
RÉPONSE  : Les frais de service pour un Prêt Express+ sur 60 jours sont de 6 % du montant, ce qui représente par exemple des frais de 9 000 FCFA pour un prêt de 150 000 FCFA.
STATUT   : preuves : DOC-03 « Les frais de service sont de 6 % du montant, soit 3 % par tr » | DOC-06 « Prêt Express+ 60 jours 6 % du montant Prêt de 150 000 FCFA : »
----------------------------------------------------------------------------------------------------
QUESTION : Quand les intérêts de l'Épargne Plus sont-ils versés ?
RÉPONSE  : Les intérêts de l'Épargne Plus sont versés chaque trimestre, à la fin des mois de mars, juin, septembre et

## <a id="interface"></a>Interface

Votre fonction `answer` se branche sur un chat Gradio, qui affiche les chunks retrouvés, les citations vérifiées, puis la réponse. Posez les questions du TP, puis des questions piège, et ouvrez les blocs pour vérifier les extraits cités.

In [ ]:
import front

demo = front.rag_chat(answer, titre="Assistant Lagune Banque", description="Votre assistant du TP3 : citations vérifiées et refus hors corpus (données fictives).",
                      exemples=QUESTIONS_METIER[:3] + QUESTIONS_PIEGES[:2])
front.launch(demo)

/Volumes/DataDisk/WORK/TRAINING/IAGEN/genai-training-participants/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Volumes/DataDisk/WORK/TRAINING/IAGEN/genai-training-participants/.venv/lib/python3.11/site-packages/gradio/queueing.py:176: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  with pd.option_context("future.no_silent_downcasting", True):
/Volumes/DataDisk/WORK/TRAINING/IAGEN/genai-training-participants/.venv/lib/python3.11/site-packages/gradio/queueing.py:178: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  pd.DataFrame(records).fillna(value=np.nan).infer_objects(copy=False)  # type: ignore
/Volumes/DataDisk/WORK/TRAINING/IAGEN/genai-training-participants/.venv/lib/python3.11/site-packages/gradio/queueing.py:176: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  with pd.option_context("future.no_s

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [26]:
try:
    assert 0.3 < SEUIL_SCORE < 0.95, "SEUIL_SCORE : un seuil entre 0,3 et 0,95 (TODO 1)"
    assert set(Citation.model_fields) == {"source", "extrait"} and set(SourcedAnswer.model_fields) == {"reponse", "citations"}, "schémas : Citation (source, extrait) et SourcedAnswer (reponse, citations) (TODO 2)"
    sortie = ask_model(question, chunks)
    assert isinstance(sortie, SourcedAnswer) and sortie.reponse, "ask_model doit renvoyer un SourcedAnswer validé (TODO 3)"
    assert sum(not r["refus"] for r in resultats) >= 7, "au moins 7 questions métier sur 10 doivent être répondues"
    assert sum(r["refus"] for r in pieges) >= 1, "au moins 1 piège sur 3 doit être refusé"
    assert all(r["citations"] for r in resultats if not r["refus"]), "une réponse non refusée porte au moins une citation vérifiée"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")

OK
